I think everything up to some point idk. I have to do this cause i really want to test the GP code on some real tickers. 

GP is only being applied to the high return portfolio. I don't think it is necessary for the low volatility portfolio to be completely honest. 

### IMPORTANT 

It is becoming clear to me that hyperoptimizing our parameters does not have a significant effect on our winrate in validation periods. Why? Because our ticker list is like 32 ish tickers. When the GP runs with the upper bound of 20, chances are the parameters that allow for 20 tickers out of the 32 to pass the indicator filter are not that hyperoptimized for returns but rather just so we can get 20 tickers pass. It is purely for this reason that I STRONGLY believe the way we weigh each ticker likely matters much more for maximizing returns than the tickers that we let through.


Play around with this, try not to break it. Test indicators. But more importantly, we need an actual weighing strategy. That along with edge case proofing the code should be our top priorities.

In [34]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime
from skopt import gp_minimize
from skopt.space import Real

### PIPELINE PART A: DATA DOWNLOAD

In [2]:
#INTITAL PORTFOLIO FILTER -
#1. needs to be company stock
#2. remove stocks not listed in us or canada
#3. remove stocks not in USD or Cad
#4. keep only stocks with avg volume>=5000 (oct 1,2025 to sep 30,2026)
#5. exclude months with <18 trading days when finding avg


#read the tickers
tickers = pd.read_csv("Tickers_Example.csv", header=None)
tickers = tickers.dropna()

eligible = []
eligible_currency = []

for i in range(len(tickers)):
    symbol = str(tickers.iloc[i, 0])
    valid = True

    try:                                   
        stock = yf.Ticker(symbol)
        info = stock.info

        #company stock?
        if info.get("quoteType") == "EQUITY":
            valid = True
        else:
            valid = False

        #listed in US/CANADA?
        valid_listing = False
        if info.get("market") == "us_market":
            valid_listing = True
        if info.get("market") == "ca_market":
            valid_listing = True
        if  valid_listing == False:
            valid = False

        #currency in usd/cad?
        valid_currency = False
        if info.get("currency") == "USD":
            valid_currency = True
        if info.get("currency") == "CAD":
            valid_currency= True
        if valid_currency == False:
            valid = False

        #exclude stocks with avgv volume <5000 (oct 1,2025 to sep 30,2026) and months with <18 trading days are excluded in avg
        if valid:
            hist = stock.history(start="2025-10-01", end="2026-09-30")
            hist.index = hist.index.strftime('%Y-%m-%d')
            dates = pd.to_datetime(hist.index)

            days_per_month = hist["Volume"].groupby([dates.year, dates.month]).count()
            volume_per_month = hist["Volume"].groupby([dates.year, dates.month]).sum()

            total_days = 0
            total_volume = 0
            for m in range(len(days_per_month)):
                if days_per_month.iloc[m] >= 18:
                    total_days = total_days + days_per_month.iloc[m]
                    total_volume = total_volume + volume_per_month.iloc[m]

            if total_days == 0:
                valid = False
            elif total_volume/total_days < 5000:
                valid = False

    except Exception:
        valid = False

    #ticker alr in list?
    already_added = False
    for j in range(len(eligible)):
        if eligible[j] == symbol:
            already_added = True

    if valid and already_added == False:
        eligible.append(symbol)
        eligible_currency.append(info.get("currency"))

#download price data for eligible tickers
all_stock = yf.download(eligible, start="2023-10-01" , end="2026-11-28" , group_by="ticker")
all_stock.index = all_stock.index.strftime('%Y-%m-%d')

stock_data = {}

for i in range(len(eligible)):
    ticker = eligible[i]

    indiv_stock = all_stock[ticker].copy()        
    indiv_stock = indiv_stock.dropna(how="all")    

    stock_data[ticker] = indiv_stock

#convert tickers in usd to cad
rate = ['CADUSD=X']
fx_tickers = yf.Tickers(' '.join(rate))

exchange_rate = []
for i in fx_tickers.tickers:
    exchange_rate.append(fx_tickers.tickers[i].history(start="2025-10-01", end="2026-11-28").Close)

cadusd = exchange_rate[0]
cadusd.index = cadusd.index.strftime('%Y-%m-%d')

for i in range(len(eligible)):
    ticker = eligible[i]
    table = stock_data[ticker]

    if eligible_currency[i] == "USD":
        table["CADUSD"] = cadusd
        table["Close_CAD"] = table["Close"] / table["CADUSD"]
        table["High_CAD"] = table["High"] / table["CADUSD"]
        table["Low_CAD"] = table["Low"] / table["CADUSD"]
    else:
        table["Close_CAD"] = table["Close"]
        table["High_CAD"] = table["High"]
        table["Low_CAD"] = table["Low"]

    stock_data[ticker] = table

print("Eligible tickers:", eligible)

for ticker in eligible:
    print("Comapny:", ticker)
    print(stock_data[ticker])


[*********************100%***********************]  35 of 35 completed


Eligible tickers: ['AAPL', 'ABBV', 'ABT', 'ACN', 'AIG', 'AMZN', 'AXP', 'BA', 'BAC', 'BB.TO', 'BIIB', 'BLK', 'BMY', 'C', 'CAT', 'CL', 'KO', 'LLY', 'LMT', 'MO', 'MRK', 'PEP', 'PFE', 'RY.TO', 'PM', 'PYPL', 'QCOM', 'SHOP.TO', 'T.TO', 'TD.TO', 'TXN', 'UNH', 'UNP', 'UPS', 'USB']
Comapny: AAPL
Price             Open        High         Low       Close      Volume  \
Date                                                                     
2023-10-02  168.806867  171.843460  168.520946  171.301208  52164500.0   
2023-10-03  169.832204  171.182906  168.412511  169.970230  49594600.0   
2023-10-04  168.678692  171.754730  168.560388  171.212479  53020300.0   
2023-10-05  171.340675  172.977283  170.246318  172.444901  48527900.0   
2023-10-06  171.350514  175.481463  170.739241  174.988510  57266700.0   
...                ...         ...         ...         ...         ...   
2026-11-16  357.130005  357.130005  351.309998  352.779999  50292200.0   
2026-11-17  352.829987  355.359985  349.670013

### PIPELINE PART B: INDICATOR FUNCTIONS + INDICATOR CALLS

In [3]:
def pull_market_cap(ticker, target_date):

  target_dt = pd.to_datetime(target_date)
  end_dt = target_dt + pd.Timedelta(days=5)

  hist = ticker.history(start=target_dt.strftime("%Y-%m-%d"), end=end_dt.strftime("%Y-%m-%d"))

  close_price = hist.iloc[0]["Close"]

  shares_history = ticker.get_shares_full(start=target_dt.strftime("%Y-%m-%d"), end=end_dt.strftime("%Y-%m-%d"))

  if shares_history is not None:
      share_count = shares_history.iloc[0]
  else:
      share_count = ticker.info.get("sharesOutstanding") #not sure what else to do but just put my hands together and pray no share split


  market_cap = close_price * share_count
  return market_cap

In [4]:
def calculate_adx(data, period=14):

    if len(data) < 2 * period:
        return pd.DataFrame(np.nan, index=data.index, columns=["adx", "DX", "+DI", "-DI", "smoothed TR"])

    data = data[["High", "Low", "Close"]].copy()

    # 1. True Range
    high_low = data["High"] - data["Low"]
    high_close = abs(data["High"] - data["Close"].shift(1))
    low_close = abs(data["Low"] - data["Close"].shift(1))

    data["Trading Range"] = high_low
    data["up gap"] = high_close
    data["down gap"] = low_close

    data["True Range"] = pd.concat(
        [high_low, high_close, low_close], axis=1
    ).max(axis=1)

    # 2. Directional Momentum
    up_move = data["High"].diff()
    down_move = -data["Low"].diff()

    pos_cond = (up_move > down_move) & (up_move > 0)
    neg_cond = (down_move > up_move) & (down_move > 0)

    data["+DM"] = np.where(pos_cond, up_move, 0)
    data["-DM"] = np.where(neg_cond, down_move, 0)

    # 3. Wilder Smoothing
    def wilder_smoothing(series, smoothing_period):
        smoothed = {}
        initial = sum(series.values[1:smoothing_period + 1])
        smoothed[smoothing_period] = initial

        for i in range(smoothing_period + 1, len(series)):
            smoothed[i] = (
                smoothed[i - 1]
                - (smoothed[i - 1] / smoothing_period)
                + series.values[i])

        return pd.Series(list(smoothed.values()), index=series.index[smoothing_period:])

    data["smoothed +DM"] = wilder_smoothing(data["+DM"], period)
    data["smoothed -DM"] = wilder_smoothing(data["-DM"], period)
    data["smoothed TR"] = wilder_smoothing(data["True Range"], period)

    # 4. Directional Index
    data["+DI"] = (data["smoothed +DM"] / data["smoothed TR"]) * 100
    data["-DI"] = (data["smoothed -DM"] / data["smoothed TR"]) * 100

    data["DX"] = (100 * abs(data["+DI"] - data["-DI"])) / (abs(data["+DI"]) + abs(data["-DI"]))

    # 5. Average Directional Index
    adx_values = {}
    initial = data["DX"].iloc[period:2 * period].mean()

    adx_values[2 * period - 1] = initial

    for i in range(2 * period, len(data)):
        adx_values[i] = ((period - 1) * adx_values[i - 1] + data["DX"].iloc[i]) / period

    data["adx"] = pd.Series(list(adx_values.values()), index=data.index[2 * period - 1:])

    data = data[["adx", "DX", "+DI", "-DI", "smoothed TR"]]

    return data

In [5]:
def rolling_price_std(data, period):

    rolling_std = data["Close"].pct_change(fill_method=None).rolling(period).std()

    return rolling_std

In [6]:
def pull_beta(ticker):

    info = ticker.info
    beta = info.get("beta")

    return beta

In [7]:
ADX_PERIODS = {"1M": 21, "3M": 63, "6M": 126}

def indicator_application(ticker, ticker_df, start_date):
    ticker = yf.Ticker(ticker)

    ticker_df["Market Cap"] = pull_market_cap(ticker, start_date)

    ticker_df["Rolling STD 63d"] = rolling_price_std(ticker_df, 63)

    ticker_df["Beta"] = pull_beta(ticker)

    for label, period in ADX_PERIODS.items():
        adx_df = calculate_adx(ticker_df, period=period)
        adx_df = adx_df.add_suffix(f" {label}")
        
        adx_df[f"DI spread {label}"] = (adx_df[f"+DI {label}"] - adx_df[f"-DI {label}"])
        ticker_df = ticker_df.drop(columns=adx_df.columns, errors="ignore")
        ticker_df = pd.concat([ticker_df, adx_df], axis=1)

    return ticker_df


In [8]:
for ticker, ticker_data in stock_data.items():
    stock_data[ticker] = indicator_application(ticker, ticker_data, "2026-11-20")

### PART C: GAUSSIAN PROCESS HYPERPARAMETER OPTIMIZATION

In [9]:
# basically the length of our competition 
HOLD_DAYS = 6 

series_by_ticker = {}
for ticker, ticker_data in stock_data.items():
  series_by_ticker[ticker] = ticker_data["Close"]

close = pd.DataFrame(series_by_ticker)

returns = close.pct_change(fill_method=None)

fwd_returns = close.shift(-HOLD_DAYS) / close - 1

print(f"Close Table Shape: {close.shape}")

Close Table Shape: (805, 35)


Condition rule setting.

For High Return Only!!!

In [23]:
HIGH_RET_RULES = {
    #"adx 6M": ">",
    #"DI spread 6M": ">",
    "adx 3M": ">",
    "DI spread 3M": ">",
    "adx 1M": ">",
    "DI spread 1M": ">",
    "Beta": ">",
    "Market Cap": "<",
}

In [24]:
def build_tables(rules):

    #essentially evert indicator in rules gets its own column 

    tables = {} 
    for col in rules: 
        col_data = {}
        for t, df in stock_data.items():
            col_data[t] = df[col]
        tables[col] = pd.DataFrame(col_data)
    return tables

high_return_tables = build_tables(HIGH_RET_RULES)
        

Instructions here:

For first_table, always put in the indicator that requires the MOST amount of days
eg. adx_6M > adx_3M

In [25]:
first_table = high_return_tables["adx 3M"]

has_indicators = first_table.notna().any(axis=1)
has_future_return = fwd_returns.notna().any(axis=1)

usable = first_table.index[has_indicators & has_future_return]

test_dates = usable[::-HOLD_DAYS][:100][::-1]

split = int(len(test_dates) * 0.7)
tune_dates = test_dates[:split]
validation_dates = test_dates[split:]

print(len(tune_dates), "tuning weeks:    ", tune_dates[0], "->", tune_dates[-1])
print(len(validation_dates), "validation weeks:", validation_dates[0], "->", validation_dates[-1])

70 tuning weeks:     2024-07-18 -> 2026-03-03
30 validation weeks: 2026-03-11 -> 2026-11-12


Which indicators pass

testing range for Gaussian Process

In [26]:
def passes(tables, rules, cutoffs, date):
    mask = pd.Series(True, index=close.columns)

    for (col, direction), cutoffs in zip(rules.items(), cutoffs):
        values = tables[col].loc[date]

        if direction == ">":
            this_rule = values > cutoffs 
        else:
            this_rule = values < cutoffs 
            
        mask = mask & this_rule

    return mask 


def make_space(tables, rules, dates):
    space = []
    for col, direction in rules.items():

        if direction == ">":
            q = [0.05, 0.90] 
        else:
            q = [0.10, 0.95]

        lo, hi = tables[col].loc[dates].stack().quantile(q)

        space.append(Real(float(lo), float(hi), name=col))

    return space


hr_space = make_space(high_return_tables, HIGH_RET_RULES, tune_dates)

scoring the tests

In [27]:

MIN_TICKERS = 10   #setting the punishment for min tickers above the min of 10. I'd rather have more than less. 
MAX_TICKERS = 20   # max 


def weekly_high_return(cutoffs, dates):
    """For each date: pick tickers, hold 6 days, record the return. NaN = too few passed."""
    out = []

    for i in dates:
        passed = passes(high_return_tables, HIGH_RET_RULES, cutoffs, i)

        if passed.sum() < MIN_TICKERS:     # not enough tickers passed this week
            out.append(np.nan)      # mark the week as failed 
            continue 


        # PLACEHOLDER PLACEHOLDER PLACEHOLDER
        # Placeholder logic:
        # If more than 20 tickers pass, take the largest 20 at equal weighting.
        picks = high_return_tables["adx 3M"].loc[i, passed].nlargest(MAX_TICKERS).index
        out.append(fwd_returns.loc[i, picks].mean())

    return pd.Series(out, index=dates)


def count_passing(tables, rules, cutoffs, dates):
    """How many tickers passed the rules on each date."""
    counts = []
    for i in dates:
        counts.append(passes(tables, rules, cutoffs, i).sum())
    return pd.Series(counts, index=dates)


def score_high_return(cutoffs, dates):
    """ONE number for these cutoffs: average weekly return, with penalties for failed weeks."""
    weekly = weekly_high_return(cutoffs, dates)
    n = count_passing(high_return_tables, HIGH_RET_RULES, cutoffs, dates)

    shortfall = (MIN_TICKERS - n).clip(lower=0)
    penalty = -0.10 - 0.01 * shortfall          # -10%, and another -1% per missing ticker

    return weekly.fillna(penalty).mean()

In [28]:
result_high_returns = gp_minimize(

    lambda p: -score_high_return(p, tune_dates),
    hr_space, 
    n_calls=100,
    n_initial_points=10,
    acq_func="EI", 
    random_state=0, 
)

best_hr = result_high_returns.x 

# no cutoff comparison 
no_cut_hr = [-np.inf if d == ">" else np.inf for d in HIGH_RET_RULES.values()]


print("best cutoffs:", dict(zip(HIGH_RET_RULES, [round(float(x), 2) for x in best_hr])))
print()


print("                tune      val")
print("tuned:      ", round(score_high_return(best_hr, tune_dates), 4),
      "  ", round(score_high_return(best_hr, validation_dates), 4))
print("no cutoffs: ", round(score_high_return(no_cut_hr, tune_dates), 4),
      "  ", round(score_high_return(no_cut_hr, validation_dates), 4))
print()


tuned_weeks = weekly_high_return(best_hr, validation_dates)
base_weeks = weekly_high_return(no_cut_hr, validation_dates)

wins = (tuned_weeks > base_weeks).sum()
print(wins, "wins out of", len(validation_dates), "validation weeks")


print("tickers passing each validation week:",
      count_passing(high_return_tables, HIGH_RET_RULES, best_hr, validation_dates).tolist())

best cutoffs: {'adx 3M': 6.02, 'DI spread 3M': -8.65, 'adx 1M': 15.15, 'DI spread 1M': -17.15, 'Beta': 0.22, 'Market Cap': 1963624493550.4}

                tune      val
tuned:       -0.0005    0.0072
no cutoffs:  -0.0021    0.0058

15 wins out of 30 validation weeks
tickers passing each validation week: [22, 21, 20, 19, 15, 18, 16, 16, 17, 16, 17, 17, 13, 16, 17, 17, 16, 14, 19, 17, 18, 16, 13, 12, 10, 12, 18, 18, 19, 20]


In [32]:
def show_passing(cutoffs, dates, show_picks=True):
    for i in dates:
        passed = passes(high_return_tables, HIGH_RET_RULES, cutoffs, i)
        names = list(passed[passed].index)
        print(f"{i}  ({len(names)} passed): {names}")

        if show_picks and len(names) >= MIN_TICKERS:
            picks = high_return_tables["adx 3M"].loc[i, passed].nlargest(MAX_TICKERS).index
            dropped = [t for t in names if t not in picks]
            print(f"     held ({len(picks)}): {list(picks)}")
            if dropped:
                print(f"     passed but cut by top-{MAX_TICKERS}: {dropped}")
        print()
latest = high_return_tables["adx 3M"].dropna(how="all").index[-1]
show_passing(best_hr, [latest])

2026-11-20  (11 passed): ['ACN', 'BLK', 'CL', 'KO', 'LLY', 'MO', 'PEP', 'PFE', 'T.TO', 'UNP', 'UPS']
     held (11): ['T.TO', 'KO', 'LLY', 'PFE', 'ACN', 'UNP', 'CL', 'UPS', 'MO', 'PEP', 'BLK']

